In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
test=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")
sub=pd.read_csv("/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi/sample_submission.csv")

In [ ]:
train

In [ ]:
test

In [ ]:
train.info()

In [ ]:
sub

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load data
# train = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/train.csv")
# test = pd.read_csv("/kaggle/input/eds-232-ocean-chemistry-prediction-for-calcofi/test.csv")

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

# Find common columns between train and test (excluding "DIC" from train)
common_columns = train.drop(columns=["DIC"]).columns.intersection(test.columns)

# Select the common columns for both train and test
X = train[common_columns]
y = train["DIC"]
test = test[common_columns]

# Replace NaNs with the column mean
# X.fillna(X.mean(), inplace=True)
# test.fillna(test.mean(), inplace=True)
X.fillna(X.mean())
test.fillna(test.mean())

# Normalize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

# Split the training data as before
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class OceanChemistryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = OceanChemistryDataset(X_train, y_train)
val_dataset = OceanChemistryDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_size):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.batch1 = nn.BatchNorm1d(128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 512)
        self.batch2 = nn.BatchNorm1d(512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(230, 360)
        self.batch3 = nn.BatchNorm1d(360)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(360, 230)
        self.batch4 = nn.BatchNorm1d(230)
        self.relu4 = nn.ReLU()
        self.fc5 = nn.Linear(512, 1)

        self.dropout = nn.Dropout(p=0.2) #0.2以上厳守

    def forward(self, x):
        x = self.fc1(x)
        x = self.batch1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.batch2(x)
        x = self.relu2(x)
        #x = self.dropout(x)
        #x = self.fc3(x)
        #x = self.batch3(x)
        #x = self.relu3(x)
        #x = self.dropout(x)
        #x = self.fc4(x)
        #x = self.batch4(x)
        #x = self.relu4(x)
        #x = self.dropout(x)
        x = self.fc5(x)
        return x

# Initialize the model
model = MLPModel(input_size=X_train.shape[1])
print(X_train.shape)


In [ ]:
import torch.optim as optim
import copy

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.9,
    patience=500,
    min_lr=1e-6
)

# Training function
def train_model(model, train_loader, val_loader, epochs=5000, patience=750):

    # Early Stoppingの初期設定
    best_val_loss = float('inf')
    best_model_state = None
    counter = 0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs.squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs.squeeze(), y_batch)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        if epoch % 100 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(
                f"Epoch {epoch+1}/{epochs}, "
                f"Train Loss: {running_loss/len(train_loader):.6f}, "
                f"Val Loss: {avg_val_loss:.6f}, "
                f"LR: {current_lr:.8f}, "
                f"Early Stop Counter: {counter}"
            )

        # 学習率を更新
        scheduler.step(avg_val_loss)
        
        # Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
        
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
        print(f"Best Validation Loss: {best_val_loss:.6f}")
        return model

# Train the model
train_model(model, train_loader, val_loader, epochs=5000)


In [ ]:
# Convert the test set into a torch tensor
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

# Set the model to evaluation mode
model.eval()

# Make predictions
with torch.no_grad():
    predictions = model(test_tensor).squeeze().numpy()

# Prepare submission
submission = pd.DataFrame({"id": range(1455, 1455 + len(predictions)), "DIC": predictions})
submission.to_csv("submission.csv", index=False)


In [ ]:
# score: 5.14182
# name: 大橋旅人
# id: 62503902